# Schur mode top-7 subnetworks and greedy information-flow trees

This notebook uses the companion script `schur_mode_greedy_trees_script.py` to:

- load the saved Schur modes from `outputs/schur_modes/schur_modes.npz` when available;
- take the top 7 contributors for each mode;
- extract their small source→receiver `M_ij` submatrix;
- pull the matching rows from `mij_netlist`;
- print a greedy directed flow tree for each mode.

Greedy tree rule: start at the largest-loading contributor, then repeatedly add the strongest available directed edge from any reached contributor to one unreached contributor. If the subnetwork is disconnected, start a new component at the next-largest unreached contributor.

In [ ]:
from pathlib import Path
import pandas as pd

from schur_mode_greedy_trees_script import (
    build_all_mode_greedy_trees,
    print_all_greedy_trees,
    print_greedy_tree,
    save_greedy_tree_outputs,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

## Configuration

Leave `MAX_MODES = None` to process every Schur mode. Set it to a small number like `3` when you only want a fast preview.

In [ ]:
MATRIX_PATH = Path("matrices/mij_matrix.csv")
NETLIST_PATH = Path("matrices/mij_netlist.csv")
OUTPUT_DIR = Path("outputs/schur_mode_greedy_trees")

TOP_N = 7
MAX_MODES = None  # use 3 for a quick preview
NORMALIZATION = "spectral_radius"
TARGET_SPECTRAL_RADIUS = 0.95
INCLUDE_SELF = True

# The script will use this saved archive if present; otherwise it recomputes Schur modes.
SCHUR_ARCHIVE_PATH = Path("outputs/schur_modes/schur_modes.npz")

## Build the top-7 subnetworks and greedy trees

In [ ]:
results = build_all_mode_greedy_trees(
    matrix_path=MATRIX_PATH,
    netlist_path=NETLIST_PATH,
    top_n=TOP_N,
    normalization=NORMALIZATION,
    target_spectral_radius=TARGET_SPECTRAL_RADIUS,
    include_self=INCLUDE_SELF,
    max_modes=MAX_MODES,
    schur_archive_path=SCHUR_ARCHIVE_PATH,
    prefer_schur_archive=True,
)

summary = save_greedy_tree_outputs(results, OUTPUT_DIR)
summary.head(10)

## Print every mode's greedy information-flow tree

In [ ]:
print_all_greedy_trees(results)

## Inspect one mode interactively

Change `MODE_TO_INSPECT` and rerun the cells below.

In [ ]:
MODE_TO_INSPECT = 0
mode_result = results[MODE_TO_INSPECT]
print_greedy_tree(mode_result)

In [ ]:
mode_result.contributors

In [ ]:
mode_result.submatrix

In [ ]:
mode_result.tree_edges

In [ ]:
mode_result.netlist_edges.head(25)

## Output files

The run above writes CSVs under `outputs/schur_mode_greedy_trees/`:

- one top-contributor table per mode;
- one top-7 source→receiver submatrix per mode;
- one filtered netlist edge table per mode;
- one greedy-tree edge table per mode;
- one summary table across modes.